# Day 2 — Step 3: 이동평균선 계산 [필수 과제 문제2]

02_preprocessing에서 저장한 데이터를 불러와  
종목별 5일 이동평균(MA5)을 계산하고 결측치를 처리합니다.

---

## 이동평균선(Moving Average, MA)이란?

최근 N일간의 종가를 매일 평균 낸 값을 선으로 이어 그린 것입니다.  
하루하루의 출렁임을 부드럽게 만들어 **추세(방향)** 를 보여줍니다.

```
날짜       close       MA5 (5일 평균)
----------------------------------------
1월 1일   100,000    NaN  ← 데이터 5일치 없음
1월 2일   102,000    NaN
1월 3일    98,000    NaN
1월 4일   105,000    NaN
1월 5일   103,000  101,600  ← (100+102+98+105+103)÷5
1월 6일   107,000  103,000  ← (102+98+105+103+107)÷5
```

> 처음 4일은 계산 불가 → **NaN(Not a Number)** 으로 표시됩니다.  
> 이 NaN을 그냥 두면 차트에 구멍이 생기고 DB 저장 시 오류가 납니다.  
> → **해당 날의 종가(close)로 대체**합니다.

---

## 종목별로 따로 계산해야 하는 이유

```
여러 종목이 하나의 DataFrame에 섞여있는 상황:

  ...  KRW-BTC  2026-01-30  90,000,000   ← BTC 마지막 날
       KRW-ETH  2026-01-01   3,100,000   ← ETH 첫째 날 ⚠️ 종목이 바뀜!

→ BTC 종가와 ETH 종가를 섞어서 평균 내면 의미 없는 숫자 생성!
→ ticker별로 나눠서 각각 계산해야 함
```

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

In [ ]:
# ── 02단계에서 저장한 데이터 불러오기 ────────────────────────────
df = pd.read_csv('ohlcv_preprocessed.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## [필수 과제 문제2] 5일 이동평균선 계산 및 결측치 처리

In [ ]:
def add_moving_average(df):
    """
    종목(ticker)별로 5일 이동평균(ma5)을 계산하고 NaN을 close로 대체합니다.

    처리 순서:
        1. groupby('ticker')로 종목별 분리
        2. 각 종목 내에서 close 기준 5일 rolling mean 계산
        3. NaN → 해당 행의 close 값으로 대체

    매개변수:
        df : ticker, date, close 컬럼이 있는 DataFrame

    반환값:
        ma5 컬럼이 추가된 DataFrame
    """
    df = df.copy()

    # ── 종목별 5일 이동평균 계산 ──────────────────────────────────
    # groupby('ticker')     : 같은 종목끼리 묶기
    # ['close']             : close 컬럼 선택
    # .transform(lambda x:) : 그룹별로 함수 적용, 원본과 행 수 동일하게 유지
    # x.rolling(5).mean()   : 해당 종목 내에서 5일 이동평균 계산
    df['ma5'] = df.groupby('ticker')['close'].transform(
        lambda x: x.rolling(5).mean()
    )

    # ── NaN을 close 값으로 대체 ───────────────────────────────────
    # fillna(df['close']) : NaN인 자리에 같은 행의 close 값을 채워 넣음
    # 예) ma5=NaN, close=100,000 → ma5=100,000
    df['ma5'] = df['ma5'].fillna(df['close'])

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_moving_average(df)

# ── NaN이 남아있는지 검증 ──────────────────────────────────────────
# .isnull().sum() : NaN 개수 확인
nan_count = df['ma5'].isnull().sum()

if nan_count == 0:
    print("✅ NaN 없음: 결측치 처리 완료!")
else:
    print(f"⚠️  NaN이 {nan_count}개 남아있습니다.")

In [ ]:
# ── 종목별 close vs ma5 비교 ─────────────────────────────────────
print("[종목별 close와 ma5 비교 (앞 7일)]")
print()

for ticker in df['ticker'].unique():
    # df['ticker'] == ticker : 해당 종목 행만 필터링
    ticker_df = df[df['ticker'] == ticker][['date', 'close', 'ma5']].head(7)
    print(f"--- {ticker} ---")
    print(ticker_df.to_string(index=False))
    print()

In [ ]:
# ── 컬럼 순서 최종 정리 ───────────────────────────────────────────
final_columns = [
    'ticker', 'date',
    'open', 'high', 'low', 'close', 'volume',
    'price_change', 'price_change_pct', 'high_low_diff',
    'ma5'
]
df = df[final_columns]

print("[최종 컬럼 순서]")
print(list(df.columns))
print()
print(df.head())

In [ ]:
# ── 다음 단계(04_visualization)로 전달 ───────────────────────────
df.to_csv('ohlcv_final.csv', index=False)
print("ohlcv_final.csv 저장 완료 → 04_visualization.ipynb에서 불러옵니다")